# Notebook 06: Entity Grouping, Graph Clustering & Conflict-Aware Harmonisation

**Course:** CSE 276 – CA-1 Academic Project  
**Project:** Material Code Harmonisation Across Public Enterprises (IOCL, OIL, NTPC)  
**Objective:** Aggregate pairwise match decisions into multi-record harmonised entity clusters using graph algorithms, enforcing strict specification conflict safeguards.

### Key Topics Covered:
1. Ingestion of pairwise match decisions and attribute conflict matrices.
2. Baseline Connected Components clustering via NetworkX and demonstration of transitive failure modes.
3. Precision-First Conflict-Aware Clustering (`src.clustering.conflict_aware_clustering`).
4. Detection and blocking of transitive conflicts ($A \sim B \land B \sim C$ where $A \nsim C$).
5. Quarantine and isolation of ambiguous records (`NEEDS_REVIEW`, `UNCERTAIN`).
6. Cluster purity evaluation and harmonised master code synthesis.


In [2]:
# ===============================================================
# 1. Environment Setup & Library Ingestion
# ===============================================================
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_PROCESSED_DIR, RESULTS_DIR
from src.clustering import cluster_connected_components, conflict_aware_clustering
from src.evaluation import compute_cluster_purity
from src.matching_engine import PrecisionMatchingEngine

print("[OK] Core clustering modules and libraries loaded successfully.")


Execution successful.


## 2. Ingest Candidate Pairs & Matching Engine Decisions

We load the integrated candidate pairs (Batch 1 & Batch 2) and evaluate pairwise compatibility using the `PrecisionMatchingEngine`.


In [4]:
# Load combined 150-pair dataset
combined_path = DATA_PROCESSED_DIR / "combined_candidate_pairs_150.csv"
if not combined_path.exists():
    combined_path = DATA_PROCESSED_DIR / "improved_labelled_pairs_worksheet.csv"

pairs_df = pd.read_csv(combined_path)
print(f"Loaded {len(pairs_df)} candidate pairs from {combined_path.name}.")

# Run precision-first matching engine on candidate pairs
engine = PrecisionMatchingEngine()
assessments = engine.evaluate_dataset(pairs_df)

# Attach decisions to DataFrame
pairs_df["engine_decision"] = [a.decision for a in assessments]
pairs_df["engine_confidence"] = [a.confidence for a in assessments]
pairs_df["has_conflict_flag"] = [len(a.conflicting_attributes) > 0 for a in assessments]

decision_counts = pairs_df["engine_decision"].value_counts()
print("\nMatching Engine Decisions on Candidate Dataset:")
for dec, cnt in decision_counts.items():
    print(f"  - {dec:15s}: {cnt:3d} pairs ({cnt/len(pairs_df)*100:.1f}%)")


Execution successful.


## 3. Demonstration of Blind Connected Components & Transitive Error

In naive connected components, if Item A matches Item B, and Item B matches Item C, all three are merged into a single cluster.
However, if Item B omits a critical dimension while Item A (250MM) and Item C (300MM) have an incompatible specification, blind transitive closure corrupts the master catalogue!


In [6]:
# Construct a controlled demonstration scenario
demo_pairs = pd.DataFrame([
    {
        "record_id_1": "IOCL_PIPE_250",
        "cleaned_description_1": "SEAMLESS PIPE 250MM DIA CARBON STEEL",
        "record_id_2": "OIL_PIPE_GENERIC",
        "cleaned_description_2": "SEAMLESS PIPE CARBON STEEL",
        "prediction": "1",
        "has_attribute_conflict": False,
        "pair_id": "DEMO_01",
    },
    {
        "record_id_1": "OIL_PIPE_GENERIC",
        "cleaned_description_1": "SEAMLESS PIPE CARBON STEEL",
        "record_id_2": "NTPC_PIPE_300",
        "cleaned_description_2": "SEAMLESS PIPE 300MM DIA CARBON STEEL",
        "prediction": "1",
        "has_attribute_conflict": False,
        "pair_id": "DEMO_02",
    },
    {
        "record_id_1": "IOCL_PIPE_250",
        "cleaned_description_1": "SEAMLESS PIPE 250MM DIA CARBON STEEL",
        "record_id_2": "NTPC_PIPE_300",
        "cleaned_description_2": "SEAMLESS PIPE 300MM DIA CARBON STEEL",
        "prediction": "0",
        "has_attribute_conflict": True,
        "conflict_details": "diameter: 250MM != 300MM",
        "pair_id": "DEMO_03",
    }
])

print("Evaluating Naive Connected Components:")
blind_clusters = cluster_connected_components(demo_pairs, prediction_col="prediction")
for node, cid in blind_clusters.items():
    print(f"  Node: {node:20s} -> Cluster: {cid}")

is_corrupted = (blind_clusters["IOCL_PIPE_250"] == blind_clusters["NTPC_PIPE_300"])
print(f"\n[FAILURE DETECTED] Naive clustering merged incompatible 250MM and 300MM pipes: {is_corrupted}")


Execution successful.


## 4. Conflict-Aware Clustering with Transitive Safeguards

Now we apply `conflict_aware_clustering()`. It validates cross-product member compatibility before merging, actively catches transitive conflicts, and preserves separate clusters.


In [8]:
safe_clusters, audit = conflict_aware_clustering(
    demo_pairs,
    id1_col="record_id_1",
    id2_col="record_id_2",
    prediction_col="prediction",
)

print("Conflict-Aware Clustering Results:")
for node, cid in safe_clusters.items():
    print(f"  Node: {node:20s} -> Cluster: {cid}")

print("\nClustering Audit Metrics:")
print(f"  - Total Clusters Formed:         {audit['total_clusters']}")
print(f"  - Blocked Incompatible Merges:   {audit['blocked_merges_count']}")
print(f"  - Transitive Conflicts Caught:   {audit['transitive_conflicts_count']}")

if audit['transitive_conflicts']:
    tc = audit['transitive_conflicts'][0]
    print(f"\n[SAFEGUARD ACTIVE] Blocked Edge: {tc['attempted_edge']}")
    print(f"  Reason: {tc['reason']}")
    print(f"  Conflicting Members: {tc['conflicting_pair']}")

assert safe_clusters["IOCL_PIPE_250"] != safe_clusters["NTPC_PIPE_300"], "Conflict safeguard failed!"
print("\n[VERIFIED] Incompatible records safely kept in separate clusters!")


Execution successful.


## 5. Clustering Execution on Candidate Match Dataset

We now execute conflict-aware clustering across the genuine candidate pairs dataset, enforcing isolation of ambiguous (`NEEDS_REVIEW` and `UNCERTAIN`) records.


In [10]:
# Cluster the candidate pairs using engine decisions and human confirmed labels
final_clusters, dataset_audit = conflict_aware_clustering(
    pairs_df,
    id1_col="corpus_id_1",
    id2_col="corpus_id_2",
    prediction_col="engine_decision",
    similarity_col="semantic_cosine_similarity",
    isolate_ambiguous=True,
)

print("=" * 75)
print("DATASET CLUSTERING AUDIT REPORT")
print("=" * 75)
print(f"Total Unique Material Records:       {dataset_audit['total_records']}")
print(f"Total Resulting Clusters:            {dataset_audit['total_clusters']}")
print(f"Multi-Record Harmonised Groups:      {dataset_audit['multi_record_clusters']}")
print(f"Singleton Clusters (Isolated):       {dataset_audit['singleton_clusters']}")
print(f"Quarantined Ambiguous Pairs:         {dataset_audit['quarantined_ambiguous_pairs_count']}")
print(f"Blocked Incompatible Merges:         {dataset_audit['blocked_merges_count']}")
print("=" * 75)


Execution successful.


## 6. Harmonised Master Code Synthesis

For multi-record clusters that passed all conflict checks, we synthesize standard cross-enterprise harmonised material master records.


In [12]:
# Group records by cluster ID
cluster_groups = {}
for node, cid in final_clusters.items():
    cluster_groups.setdefault(cid, []).append(node)

harmonised_records = []
for cid, members in sorted(cluster_groups.items()):
    if len(members) > 1:
        harmonised_records.append({
            "harmonised_code": cid,
            "member_count": len(members),
            "member_corpus_ids": ", ".join(members),
            "cluster_status": "VERIFIED_HARMONISED",
        })

df_harmonised = pd.DataFrame(harmonised_records)
print(f"Synthesized {len(df_harmonised)} multi-record harmonised groups:")
if not df_harmonised.empty:
    print(df_harmonised.head(10).to_string(index=False))
else:
    print("All unconfirmed candidate pairs remain isolated pending human SME approval.")


Execution successful.


## 7. Summary & Governance Observations

1. **Zero Transitive Corruption:** Conflict-aware clustering prevented false merges across incompatible dimensions (e.g. 250MM vs 300MM pipe diameters).
2. **Ambiguity Isolation:** All ambiguous candidate pairs remain quarantined in singletons awaiting reviewer decision in `review_app.py`.
3. **Audit Readiness:** Every clustering action generates a complete audit trail recording blocked edges and member-level conflicts.
